In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import re

# ✅ run 폴더
run_dir = Path('..').resolve()

print("run_dir exists:", run_dir.exists(), run_dir)


run_dir exists: True /home/dibaeck/workspace/project_IR_sLM_MAS/runs/exp2_qwen2p5_policy_v0_smoke_20260331_120344


In [2]:
trials_path = run_dir / "trials.jsonl"

print("trials exists:", trials_path.exists(), trials_path)

rows = []
with open(trials_path, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

print("Total rows:", len(rows))

trials exists: True /home/dibaeck/workspace/project_IR_sLM_MAS/runs/exp2_qwen2p5_policy_v0_smoke_20260331_120344/trials.jsonl
Total rows: 123


In [3]:
# 컬럼 확인
df = pd.DataFrame(rows)

print(df.shape)
df.head()

(123, 43)


,task_id,trial_id,attempt_index,model,prompt_hash,diff,edit_script,patch_lines_added,patch_lines_removed,files_changed,...,test_command,generated_diff,exception,policy_enabled,trigger_error_type,trigger_signature,policy_action,policy_params,final_selected,terminated_by_policy
0,astropy__astropy-12907,0,0,Qwen/Qwen2.5-Coder-7B-Instruct,1d006f36813d2fc095bc8da160e7e87abfd8d05ee8957d...,,,0,0,0,...,,,"RuntimeError('sLM call failed (provider=vllm, ...",True,,,INITIAL,"{'retry_max_files': 2, 'context_strategy': 'de...",False,False
1,astropy__astropy-14182,0,0,Qwen/Qwen2.5-Coder-7B-Instruct,323a3b727d26875c53e665ce8dcd10f792793ee5799a7a...,,,0,0,0,...,,,"RuntimeError('sLM call failed (provider=vllm, ...",True,,,INITIAL,"{'retry_max_files': 2, 'context_strategy': 'de...",False,False
2,astropy__astropy-14365,0,0,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,,"{\n ""edits"": [\n {\n ""op"": ""replace_r...",0,0,0,...,,,FileNotFoundError('astropy/io/ascii/QDP.py'),True,,,INITIAL,"{'retry_max_files': 2, 'context_strategy': 'de...",False,False
3,astropy__astropy-14365,0,1,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,,"{\n ""edits"": [\n {\n ""op"": ""replace_r...",0,0,0,...,,,FileNotFoundError('astropy/io/ascii/QDP.py'),True,APPLY_FAIL,edit_apply_path_missing,RETRY_EXPAND_FILES,"{'action': 'RETRY_EXPAND_FILES', 'retry_max_fi...",False,False
4,astropy__astropy-14995,0,0,Qwen/Qwen2.5-Coder-7B-Instruct,fd14cad64dd3edbd82b7592d7f25cd7092f88230923805...,,"{\n ""edits"": [\n {\n ""op""...",0,0,0,...,,,FileNotFoundError('astropy/nddata/_core.py'),True,,,INITIAL,"{'retry_max_files': 2, 'context_strategy': 'de...",False,False


In [4]:
# (task_id, trial_id) 기준으로 묶기
grouped = {}

for _, r in df.iterrows():
    key = (r["task_id"], r["trial_id"])
    grouped.setdefault(key, {})
    grouped[key][r["attempt_index"]] = r

print("Total grouped tasks:", len(grouped))

Total grouped tasks: 100


In [5]:
apply_recovered = 0
parse_recovered = 0
initial_pred_ready = 0

apply_total = 0
parse_total = 0

for key, attempts in grouped.items():
    a0 = attempts.get(0)
    a1 = attempts.get(1)

    if a0 is None:
        continue

    et0 = a0["error_type"]

    # 1️⃣ attempt0 already success
    if et0 == "PRED_READY":
        initial_pred_ready += 1

    # 2️⃣ APPLY_FAIL → PRED_READY
    if et0 == "APPLY_FAIL":
        apply_total += 1
        if a1 is not None and a1["error_type"] == "PRED_READY":
            apply_recovered += 1

    # 3️⃣ EDIT_PARSE_FAIL → PRED_READY
    if et0 == "EDIT_PARSE_FAIL":
        parse_total += 1
        if a1 is not None and a1["error_type"] == "PRED_READY":
            parse_recovered += 1

print("===== RESULT =====")
print("initial PRED_READY:", initial_pred_ready)
print()

print("APPLY_FAIL total:", apply_total)
print("APPLY_FAIL → PRED_READY:", apply_recovered)
if apply_total > 0:
    print("APPLY recovery rate:", round(apply_recovered / apply_total, 4))
print()

print("EDIT_PARSE_FAIL total:", parse_total)
print("EDIT_PARSE_FAIL → PRED_READY:", parse_recovered)
if parse_total > 0:
    print("PARSE recovery rate:", round(parse_recovered / parse_total, 4))
    
# task= 50
# APPLY_FAIL 10개 중에 2개 retry해서 넘김

===== RESULT =====
initial PRED_READY: 73

APPLY_FAIL total: 20
APPLY_FAIL → PRED_READY: 2
APPLY recovery rate: 0.1

EDIT_PARSE_FAIL total: 3
EDIT_PARSE_FAIL → PRED_READY: 1
PARSE recovery rate: 0.3333


In [6]:
df_attempt1 = df[df["attempt_index"] == 1]

print(df_attempt1["policy_action"].value_counts())

policy_action
RETRY_EXPAND_FILES          20
RETRY_SCHEMA_CONSTRAINED     3
Name: count, dtype: int64


In [7]:
examples = []

for key, attempts in grouped.items():
    a0 = attempts.get(0)
    a1 = attempts.get(1)

    if a0 is None or a1 is None:
        continue

    if a0["error_type"] == "APPLY_FAIL" and a1["error_type"] == "PRED_READY":
        examples.append((key, a0, a1))

print("Recovered APPLY cases:", len(examples))

for i, (key, a0, a1) in enumerate(examples[:5]):
    print("\n--- Example", i, "---")
    print("Task:", key)
    print("Attempt0:", a0["error_type"], a0["signature"])
    print("Attempt1:", a1["error_type"], a1["signature"])
    print("Policy:", a1["policy_action"])

Recovered APPLY cases: 2

--- Example 0 ---
Task: ('astropy__astropy-14995', 0)
Attempt0: APPLY_FAIL edit_apply_path_missing
Attempt1: PRED_READY ready_for_harness
Policy: RETRY_EXPAND_FILES

--- Example 1 ---
Task: ('django__django-14752', 0)
Attempt0: APPLY_FAIL edit_apply_range_oob
Attempt1: PRED_READY ready_for_harness
Policy: RETRY_EXPAND_FILES
